# 📚 StuLearn RAG Pipeline using LlamaIndex

## Overview

This notebook demonstrates how to build a **Retrieval-Augmented Generation (RAG)** pipeline for **StuLearn**, an AI-powered study assistant that answers questions using uploaded academic documents.

The pipeline combines:

- **LlamaIndex** for document indexing and retrieval
- **ChromaDB** as the vector database
- **OpenRouter** for embedding generation and LLM inference
- **Custom Prompt Engineering** to minimize hallucinations and generate grounded responses

---

## Pipeline Architecture


Documents (PDFs)
        │
        ▼
Load Documents
        │
        ▼
Sentence Chunking
        │
        ▼
Generate Embeddings
        │
        ▼
Store in ChromaDB
        │
        ▼
Retrieve Relevant Chunks
        │
        ▼
Custom Prompt
        │
        ▼
OpenRouter LLM
        │
        ▼
Grounded Answer


---

## Objectives

- Load research papers from a local directory
- Split documents into meaningful chunks
- Generate semantic embeddings
- Store embeddings in a persistent vector database
- Retrieve the most relevant chunks for a user query
- Generate accurate answers strictly based on retrieved context

---

**Author:** Hrishikesh Sanap

**Project:** Student Learn – AI-Powered Study Assistant

Imports



In [4]:
# ==========================================================
# Import Required Libraries
# ==========================================================

import os

from dotenv import load_dotenv

# ---------- LlamaIndex ----------
from llama_index.core import (
    SimpleDirectoryReader,
    VectorStoreIndex,
    Settings,
    PromptTemplate,
)

from llama_index.core.node_parser import SentenceSplitter
from llama_index.embeddings.openai import OpenAIEmbedding
from llama_index.llms.openai_like import OpenAILike

# ---------- Chroma ----------
import chromadb
from llama_index.vector_stores.chroma import ChromaVectorStore

In [5]:
# ==========================================================
# Load Environment Variables
# ==========================================================

load_dotenv()

OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")

if not OPENROUTER_API_KEY:
    raise ValueError("OPENROUTER_API_KEY not found in .env file")

In [6]:
# ==========================================================
# Project Configuration
# ==========================================================

DATA_DIRECTORY = "../data"

CHROMA_DB_PATH = "../storage/chroma_db"

COLLECTION_NAME = "student_notes"

EMBEDDING_MODEL = "text-embedding-3-large"

LLM_MODEL = "openrouter/free"

CHUNK_SIZE = 512
CHUNK_OVERLAP = 100

TOP_K = 5

In [7]:
# ==========================================================
# Load Documents
# ==========================================================

documents = SimpleDirectoryReader(
    input_dir=DATA_DIRECTORY
).load_data()

print(f"Loaded {len(documents)} document(s).")

2026-07-30 00:47:23,119 - WARNING - Ignoring wrong pointing object 6 0 (offset 0)
2026-07-30 00:47:23,120 - WARNING - Ignoring wrong pointing object 8 0 (offset 0)
2026-07-30 00:47:23,121 - WARNING - Ignoring wrong pointing object 10 0 (offset 0)
2026-07-30 00:47:23,121 - WARNING - Ignoring wrong pointing object 12 0 (offset 0)
2026-07-30 00:47:23,123 - WARNING - Ignoring wrong pointing object 14 0 (offset 0)
2026-07-30 00:47:23,124 - WARNING - Ignoring wrong pointing object 16 0 (offset 0)
2026-07-30 00:47:23,125 - WARNING - Ignoring wrong pointing object 18 0 (offset 0)
2026-07-30 00:47:23,126 - WARNING - Ignoring wrong pointing object 20 0 (offset 0)
2026-07-30 00:47:23,127 - WARNING - Ignoring wrong pointing object 22 0 (offset 0)
2026-07-30 00:47:23,128 - WARNING - Ignoring wrong pointing object 24 0 (offset 0)
2026-07-30 00:47:23,129 - WARNING - Ignoring wrong pointing object 36 0 (offset 0)
2026-07-30 00:47:23,130 - WARNING - Ignoring wrong pointing object 48 0 (offset 0)


Loaded 42 document(s).


In [8]:
# ==========================================================
# Split Documents into Chunks
# ==========================================================

splitter = SentenceSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
)

nodes = splitter.get_nodes_from_documents(documents)

print(f"Generated {len(nodes)} text chunks.")

Generated 124 text chunks.


In [9]:
# ==========================================================
# Configure Embedding Model
# ==========================================================

embed_model = OpenAIEmbedding(
    api_key=OPENROUTER_API_KEY,
    api_base="https://openrouter.ai/api/v1",
    model=EMBEDDING_MODEL,
)

Settings.embed_model = embed_model

In [10]:
# ==========================================================
# Create Persistent Chroma Database
# ==========================================================

chroma_client = chromadb.PersistentClient(
    path=CHROMA_DB_PATH
)

collection = chroma_client.get_or_create_collection(
    COLLECTION_NAME
)

vector_store = ChromaVectorStore(
    chroma_collection=collection
)

In [11]:
# ==========================================================
# Create Persistent Chroma Database
# ==========================================================

chroma_client = chromadb.PersistentClient(
    path=CHROMA_DB_PATH
)

collection = chroma_client.get_or_create_collection(
    COLLECTION_NAME
)

vector_store = ChromaVectorStore(
    chroma_collection=collection
)

In [12]:
# ==========================================================
# Build Vector Index
# ==========================================================

index = VectorStoreIndex(
    nodes,
    vector_store=vector_store,
    embed_model=embed_model,
)

print("Vector index created successfully.")

2026-07-30 00:49:09,845 - INFO - HTTP Request: POST https://openrouter.ai/api/v1/embeddings "HTTP/1.1 200 OK"
2026-07-30 00:49:11,646 - INFO - HTTP Request: POST https://openrouter.ai/api/v1/embeddings "HTTP/1.1 200 OK"


Vector index created successfully.


In [13]:
# ==========================================================
# Configure Retriever
# ==========================================================

retriever = index.as_retriever(
    similarity_top_k=TOP_K
)

In [14]:
# ==========================================================
# Test Retrieval
# ==========================================================

query = "What is performance evaluation?"

results = retriever.retrieve(query)

print(f"Retrieved {len(results)} chunks\n")

for i, node in enumerate(results, start=1):
    print("=" * 80)
    print(f"Chunk {i}")
    print(node.text[:500])
    print()

2026-07-30 00:49:36,580 - INFO - HTTP Request: POST https://openrouter.ai/api/v1/embeddings "HTTP/1.1 200 OK"


Retrieved 5 chunks

Chunk 1
15. Flegl, M., Depoo, L., & Alcázar, M. (2022). The impact of employees’ training on their performance improvements. Quality Innovation Prosperity, 26(1), 70-89. 16. Sal, A., & Raja, M. (2016). The impact of training and development on employees' performance and productivity. International Journal of Management Sciences and Business Research, 5(7). 17. Hafeez, U., & Akbar, W. (2015). Impact of training on employees performance (Evidence from pharmaceutical companies in Karachi, Pakistan). Busine

Chunk 2
SUS calculation
To rescale all values as 0–4, 1 was subtracted from the 
participant’s response for odd-numbered items, while 
for even-numbered questions, the participant’s response 
was subtracted from 5. The converted responses were 
summed for each user, and the total was multiplied by 
2.5 to convert the range of possible values from 0–40 to 
0–100 (Sauro 2011).
The overall SUS score success threshold (i.e., the mini
-
mum acceptable point at which user

In [15]:
# ==========================================================
# Configure LLM
# ==========================================================

llm = OpenAILike(
    model=LLM_MODEL,
    api_key=OPENROUTER_API_KEY,
    api_base="https://openrouter.ai/api/v1",
    is_chat_model=True,
    timeout=300,
)

In [17]:
from llama_index.core import PromptTemplate

rag_prompt = PromptTemplate("""
You are StuLearn, an AI-powered research and study assistant.

Your task is to answer questions ONLY using the provided context.

Rules:
1. Never use outside knowledge.
2. If the answer is not present, reply:
   "I could not find this information in the uploaded documents."
3. Combine information from multiple retrieved chunks whenever necessary.
4. Give detailed, well-structured answers.
5. Use headings and bullet points where appropriate.
6. Explain technical terms in simple language.
7. Do not hallucinate or guess.
8. If multiple uploaded papers discuss the topic, mention that the answer combines information from multiple papers.

-----------------------
Context
-----------------------
{context_str}

-----------------------
Question
-----------------------
{query_str}

Provide a comprehensive answer:
""")

In [18]:
query_engine = index.as_query_engine(
    llm=llm,
    similarity_top_k=TOP_K,
    text_qa_template=rag_prompt,
    streaming=True,
)

In [21]:
query = "What role does machine learning play in the system?"

retrieved_nodes = retriever.retrieve(query)

print(f"Retrieved {len(retrieved_nodes)} chunks\n")

for i, node in enumerate(retrieved_nodes, start=1):
    print("=" * 80)
    print(f"Chunk {i}")
    print(f"Score : {node.score:.4f}")
    print("-" * 80)
    print(node.text[:600])
    print()

2026-07-30 00:52:49,077 - INFO - HTTP Request: POST https://openrouter.ai/api/v1/embeddings "HTTP/1.1 200 OK"


Retrieved 5 chunks

Chunk 1
Score : 0.3957
--------------------------------------------------------------------------------
44. Thatcher C, Acharya S. Pharmaceutical uses of blockchain technology. In:
2018 IEEE International Conference on Advanced Networks and Tele-
communications Systems (ANTS), pp. 1–6. IEEE; December 16–19, 2018;
Indore, India.
45. Chakraborty S, Aich S, Kim H-C. A secure healthcare system design
framework using blockchain technology. In: 2019 21st International Con-
ference on Advanced Communication Technology (ICACT), pp. 260–4.
IEEE; February 17–20, 2019; PyeongChang, Korea (South).
46. Li P, Nelson SD, Malin BA, et al. DMMS: a decentralized blockchain led-
ger for the management of medicatio

Chunk 2
Score : 0.3783
--------------------------------------------------------------------------------
Additionally, there is significant potential for AI-Driven Sensitivity Categorization, where machine learning algorithms automatically identify and flag "Privacy Tier" me

In [22]:
context = "\n\n".join(
    [node.text for node in retrieved_nodes]
)

print(context)

44. Thatcher C, Acharya S. Pharmaceutical uses of blockchain technology. In:
2018 IEEE International Conference on Advanced Networks and Tele-
communications Systems (ANTS), pp. 1–6. IEEE; December 16–19, 2018;
Indore, India.
45. Chakraborty S, Aich S, Kim H-C. A secure healthcare system design
framework using blockchain technology. In: 2019 21st International Con-
ference on Advanced Communication Technology (ICACT), pp. 260–4.
IEEE; February 17–20, 2019; PyeongChang, Korea (South).
46. Li P, Nelson SD, Malin BA, et al. DMMS: a decentralized blockchain led-
ger for the management of medication histories.BHTY 2018; 2: 1.
47. Aldughayﬁq B, Sampalli S. A framework to lower the risk of medication
prescribing and dispensing errors: a usability study of an NFC-based mo-
bile application. Int J Med Inform 2021; 153: 104509.
48. Leo M, Sharma S, Maddulety K. Machine learning in banking risk man-
agement: a literature review.Risks 2019; 7 (1): 29.
49. Simeone O. A very brief introduction to ma

In [23]:
import tiktoken

encoding = tiktoken.get_encoding("cl100k_base")

context_tokens = len(
    encoding.encode(context)
)

print(f"Context Tokens : {context_tokens}")

Context Tokens : 1891


In [24]:
query_tokens = len(
    encoding.encode(query)
)

print(f"Query Tokens : {query_tokens}")

Query Tokens : 10


In [25]:
full_prompt = f"""
Context

{context}

Question

{query}
"""

prompt_tokens = len(
    encoding.encode(full_prompt)
)

print(prompt_tokens)

1906


In [26]:
response = query_engine.query(query)

answer = str(response)

answer_tokens = len(
    encoding.encode(answer)
)

print(answer_tokens)

2026-07-30 00:54:09,574 - INFO - HTTP Request: POST https://openrouter.ai/api/v1/embeddings "HTTP/1.1 200 OK"
2026-07-30 00:54:11,058 - INFO - HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"


510


In [27]:
print("=" * 60)

print(f"Query Tokens     : {query_tokens}")
print(f"Context Tokens   : {context_tokens}")
print(f"Answer Tokens    : {answer_tokens}")

print(f"Total Tokens     : {query_tokens + context_tokens + answer_tokens}")

Query Tokens     : 10
Context Tokens   : 1891
Answer Tokens    : 510
Total Tokens     : 2411


In [28]:
import time

start = time.perf_counter()

results = retriever.retrieve(query)

retrieval_time = time.perf_counter() - start

print(f"Retrieval Time : {retrieval_time:.3f} sec")

2026-07-30 00:54:55,910 - INFO - HTTP Request: POST https://openrouter.ai/api/v1/embeddings "HTTP/1.1 200 OK"


Retrieval Time : 0.954 sec


In [29]:
start = time.perf_counter()

response = query_engine.query(query)

generation_time = time.perf_counter() - start

print(f"Generation Time : {generation_time:.3f} sec")

2026-07-30 00:55:09,931 - INFO - HTTP Request: POST https://openrouter.ai/api/v1/embeddings "HTTP/1.1 200 OK"
2026-07-30 00:55:11,369 - INFO - HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"


Generation Time : 4.196 sec


============================================================
RAG Evaluation Summary
============================================================

Query
------------------------------------------------------------
What role does machine learning play in the system?

Retrieved Chunks     : 5
Context Tokens       : 1867
Question Tokens      : 11
Answer Tokens        : 452
Total Tokens         : 2330

Retrieval Time       : 0.09 sec
Generation Time      : 1.74 sec

Embedding Model      : text-embedding-3-large
LLM                  : openrouter/free

============================================================